# 01 Long-tailed distribution

Workflow:
1. Read filtered detections and compute **Clerkin-style frame prevalence**.
2. Build and save a category-level intermediate summary (one category per row).
3. Read intermediate summary for all downstream analyses and figures.
4. Fit and save power-law summaries for overall and semantic-specific distributions.

Frame prevalence = unique frames with >=1 detection of category / total unique frames in pool.
- `valid129`: full infant-view frame pool (0.27-filtered detections).
- `valid85`: annotation / VQA frame pool (rater-validated exemplar frames).

Run conventions (aligned with other preprint notebooks):
- Set `CATEGORY_SET` in the setup cell to `"valid129"` (default) or `"valid85"`.
- Outputs go to `analysis/manuscript-2026/main_results_valid129s_04302026/` or `analysis/manuscript-2026/supplemental_results_valid85cats_04302026/`.
- Tables are saved under each run's `results/`; figures under `figures/`.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]

# Capsule-aware paths (Code Ocean: code/; monorepo: analysis/)
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
for _cand in (_HERE, _HERE / "code" / "manuscript-2026", _HERE / "analysis" / "manuscript-2026"):
    if (_cand / "manuscript_config.py").is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

from manuscript_config import DATA_DIR, MANUSCRIPT_DIR, PREPRINT_DIR, PROJECT_ROOT

# Back-compat aliases used later in this notebook
ANALYSIS_DIR = MANUSCRIPT_DIR.parent
ROOT = PROJECT_ROOT

# Set this once at the top of the notebook.
# Supported values: "valid129" (default) or "valid85"
CATEGORY_SET = "valid129"

CATEGORY_FILES = {
    "valid85": DATA_DIR / "included_categories_valid85.txt",
    "valid129": DATA_DIR / "included_categories_valid129.txt",
}
if CATEGORY_SET not in CATEGORY_FILES:
    raise ValueError(
        f"Unsupported CATEGORY_SET: {CATEGORY_SET!r} (expected one of {sorted(CATEGORY_FILES)})"
    )
INCLUDED_CATEGORIES_TXT = CATEGORY_FILES[CATEGORY_SET]

if CATEGORY_SET == "valid129":
    OUTPUT_RUN_ROOT = PREPRINT_DIR / "main_results_valid129s_04302026"
else:
    OUTPUT_RUN_ROOT = PREPRINT_DIR / "supplemental_results_valid85cats_04302026"

RESULTS_DIR = OUTPUT_RUN_ROOT / "results"
FIGURES_DIR = OUTPUT_RUN_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD_TOKEN = "0.27"
FILE_SUFFIX = f"filtered-{THRESHOLD_TOKEN}_{CATEGORY_SET}"
OUTPUT_INTERMEDIATE = RESULTS_DIR / f"long_tailed_dist_prop_included_categories_{FILE_SUFFIX}.csv"

print(f"[01_long_tailed_distribution] CATEGORY_SET={CATEGORY_SET!r}")
print(f"Included categories txt: {INCLUDED_CATEGORIES_TXT}")
print(f"Run root: {OUTPUT_RUN_ROOT}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Figures dir: {FIGURES_DIR}")

# Color palette aligned with ccn-2025/long_tailed_distribution_163cats.py
CDI_SEMANTIC_ORDER = [
    "animals", "body_parts", "clothing", "food_drink", "furniture_rooms",
    "household", "outside", "people", "toys", "vehicles", "other",
]
CDI_SEMANTIC_COLORS = {
    "animals": "#4DB8A8",
    "body_parts": "#E87A5F",
    "clothing": "#9B7EC8",
    "food_drink": "#E8A54C",
    "furniture_rooms": "#6BAB7A",
    "household": "#D97B9E",
    "outside": "#5B9BD5",
    "people": "#E8C44C",
    "toys": "#B07CC8",
    "vehicles": "#6BA3D5",
    "other": "#8B9A9E",
}


def _apply_axis_style(ax):
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis="both", labelsize=13, width=1.2)


In [ ]:
# CATEGORY_SET is configured in the setup cell above.
# No additional override cell is needed.


In [ ]:
# 1) Build category-level intermediate summary (Clerkin-style frame prevalence).
#    Inclusion is defined by data/included_categories_{valid85|valid129}.txt via CATEGORY_SET.

included_categories = [
    line.strip().lower()
    for line in INCLUDED_CATEGORIES_TXT.read_text().splitlines()
    if line.strip()
]

df_cdi = pd.read_csv(CDI_CSV, usecols=["uni_lemma", "category"])
df_cdi["uni_lemma"] = df_cdi["uni_lemma"].astype(str).str.strip().str.lower()
df_cdi["category"] = df_cdi["category"].astype(str).str.strip().str.lower()
lemma_to_semantic = (
    df_cdi.drop_duplicates(subset=["uni_lemma"], keep="first")
    .set_index("uni_lemma")["category"]
    .to_dict()
)

df_det = load_detections(FRAME_DATA_CSV)
df_cat, pool_label = build_frame_prevalence_table(
    df_det,
    CATEGORY_SET,
    included_categories,
    lemma_to_semantic,
)
df_cat = df_cat.sort_values("proportion", ascending=False).reset_index(drop=True)

df_cat.to_csv(OUTPUT_INTERMEDIATE, index=False)
sync_public_frequency_tables(df_cat, CATEGORY_SET, DATA_DIR, threshold_token=THRESHOLD_TOKEN)

print(f"Saved intermediate file: {OUTPUT_INTERMEDIATE}")
print(f"Frame-prevalence pool: {pool_label}")
print(f"Included categories: {len(included_categories)}")
print(f"Total frames in pool: {int(df_cat['total_frames'].iloc[0])}")
df_cat.head(10)


In [ ]:
# 2) Plot long-tailed distribution:
#    - Combined panel with top-50 categories (color-coded by cdi_semantic)
#    - Subplots for each cdi semantic category (top 10 within each)
#    - Additional figure for all included categories

# Read from intermediate so downstream steps always use the same source
df_plot = pd.read_csv(OUTPUT_INTERMEDIATE)
df_plot["category"] = df_plot["category"].astype(str).str.strip().str.lower()
df_plot["cdi_semantic"] = df_plot["cdi_semantic"].astype(str).str.strip().str.lower()
df_plot = df_plot.sort_values("proportion", ascending=False).reset_index(drop=True)

# Keep only semantics that exist in this dataset, then sort subplot order by total frequency
semantic_present = [s for s in CDI_SEMANTIC_ORDER if s in set(df_plot["cdi_semantic"])]
if len(semantic_present) == 0:
    semantic_present = sorted(df_plot["cdi_semantic"].dropna().unique().tolist())

semantic_rank = (
    df_plot[df_plot["cdi_semantic"].isin(semantic_present)]
    .groupby("cdi_semantic", as_index=False)["proportion"]
    .sum()
    .sort_values("proportion", ascending=False)
)
semantic_present = semantic_rank["cdi_semantic"].tolist()

# Small ordering override requested for presentation consistency.
if "furniture_rooms" in semantic_present and "household" in semantic_present:
    semantic_present.remove("furniture_rooms")
    household_idx = semantic_present.index("household")
    semantic_present.insert(household_idx, "furniture_rooms")

# Figure layout: top row full-width + semantic subplots beneath
# Use a balanced grid for semantic panels so there is no "orphan" subplot on the last row.
n_sem = len(semantic_present)
if n_sem <= 4:
    n_cols = max(1, n_sem)
elif n_sem <= 6:
    n_cols = 3
elif n_sem <= 8:
    n_cols = 4
else:
    n_cols = 5
n_rows_sub = int(np.ceil(n_sem / n_cols)) if n_sem > 0 else 1

fig = plt.figure(figsize=(18, 4 + 3.2 * n_rows_sub), constrained_layout=True)
gs = GridSpec(1 + n_rows_sub, n_cols, figure=fig, height_ratios=[1.4] + [1] * n_rows_sub)

# Top panel: top-50 overall (no legend; semantic labels are in subplot titles)
ax_top = fig.add_subplot(gs[0, :])
top50 = df_plot.head(50).copy()
colors_50 = [CDI_SEMANTIC_COLORS.get(s, CDI_SEMANTIC_COLORS["other"]) for s in top50["cdi_semantic"]]
x50 = np.arange(len(top50))
ax_top.bar(x50, top50["proportion"], color=colors_50, edgecolor="none", width=0.8)
ax_top.set_xticks(x50)
ax_top.set_xticklabels(top50["category"], rotation=45, ha="right", fontsize=10)
ax_top.set_ylabel(FRAME_PREVALENCE_YLABEL, fontsize=14)
ax_top.set_title("Top 50 categories overall (colored by CDI semantic category)")
_apply_axis_style(ax_top)

# Semantic subplots: top 10 within each CDI semantic category
# Fix identical bar width and identical x-span across subplots for visual consistency.
for idx, sem in enumerate(semantic_present):
    row = 1 + idx // n_cols
    col = idx % n_cols
    ax = fig.add_subplot(gs[row, col])

    sub = (
        df_plot[df_plot["cdi_semantic"] == sem]
        .sort_values("proportion", ascending=False)
        .head(10)
    )

    x = np.arange(len(sub))
    color = CDI_SEMANTIC_COLORS.get(sem, CDI_SEMANTIC_COLORS["other"])
    ax.bar(x, sub["proportion"], color=color, edgecolor="none", width=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["category"], rotation=45, ha="right", fontsize=10)
    ax.set_xlim(-0.5, 9.5)  # same x-range for every subplot to keep bar size consistent
    if idx == 0:
        ax.set_ylabel(FRAME_PREVALENCE_YLABEL, fontsize=14)
    else:
        ax.set_ylabel("")
    ax.set_title(sem.replace("_", " "), color=color, fontsize=12, fontweight="bold")
    _apply_axis_style(ax)

# Hide any unused subplot slots
total_slots = n_rows_sub * n_cols
for j in range(n_sem, total_slots):
    row = 1 + j // n_cols
    col = j % n_cols
    ax_unused = fig.add_subplot(gs[row, col])
    ax_unused.axis("off")

out_png = FIGURES_DIR / f"long_tailed_top50_plus_semantic_subplots_{FILE_SUFFIX}.png"
out_pdf = FIGURES_DIR / f"long_tailed_top50_plus_semantic_subplots_{FILE_SUFFIX}.pdf"
fig.savefig(out_png, dpi=150, bbox_inches="tight")
fig.savefig(out_pdf, bbox_inches="tight")
plt.show()

print(f"Saved figure: {out_png}")
print(f"Saved figure: {out_pdf}")

# Additional figure: all included categories (single panel)
fig_all, ax_all = plt.subplots(figsize=(26, 8), constrained_layout=True)
colors_all = [CDI_SEMANTIC_COLORS.get(s, CDI_SEMANTIC_COLORS["other"]) for s in df_plot["cdi_semantic"]]
x_all = np.arange(len(df_plot))
ax_all.bar(x_all, df_plot["proportion"], color=colors_all, edgecolor="none", width=0.8)
ax_all.set_xticks(x_all)
ax_all.set_xticklabels(df_plot["category"], rotation=70, ha="right", fontsize=10)
ax_all.set_ylabel(FRAME_PREVALENCE_YLABEL, fontsize=14)
ax_all.set_title("All included categories (colored by CDI semantic category)")
_apply_axis_style(ax_all)

legend_handles_all = [
    Patch(facecolor=CDI_SEMANTIC_COLORS[k], label=k.replace("_", " "))
    for k in semantic_present
]
ax_all.legend(
    handles=legend_handles_all,
    ncol=min(6, len(legend_handles_all)),
    frameon=False,
    fontsize=12,
    loc="upper right",
)

out_all_png = FIGURES_DIR / f"long_tailed_all_included_categories_{FILE_SUFFIX}.png"
out_all_pdf = FIGURES_DIR / f"long_tailed_all_included_categories_{FILE_SUFFIX}.pdf"
fig_all.savefig(out_all_png, dpi=150, bbox_inches="tight")
fig_all.savefig(out_all_pdf, bbox_inches="tight")
plt.show()

print(f"Saved figure: {out_all_png}")
print(f"Saved figure: {out_all_pdf}")


In [ ]:
# 4) Power-law fitting on rank-frequency distributions
#    - Overall: all included categories
#    - Per semantic category: categories within each CDI semantic label

fit_df = pd.read_csv(OUTPUT_INTERMEDIATE)


def fit_power_law_rank(values: np.ndarray, min_points: int = 3) -> dict:
    vals = np.asarray(values, dtype=np.float64)
    vals = vals[np.isfinite(vals)]
    vals = vals[vals > 0]

    vals = np.sort(vals)[::-1]
    n = vals.size
    if n < min_points:
        return {
            "n_points": int(n),
            "alpha": np.nan,
            "intercept_log": np.nan,
            "coef_c": np.nan,
            "r2_log": np.nan,
            "rmse_log": np.nan,
        }

    ranks = np.arange(1, n + 1, dtype=np.float64)
    x = np.log(ranks)
    y = np.log(vals)

    slope, intercept = np.polyfit(x, y, deg=1)
    y_hat = slope * x + intercept

    ss_res = float(np.sum((y - y_hat) ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
    rmse = float(np.sqrt(np.mean((y - y_hat) ** 2)))

    return {
        "n_points": int(n),
        # y = log(C) - alpha * log(rank) => slope = -alpha
        "alpha": float(-slope),
        "intercept_log": float(intercept),
        "coef_c": float(np.exp(intercept)),
        "r2_log": float(r2),
        "rmse_log": rmse,
    }


# Build fit summary table
rows = []

# Overall distribution
overall_vals = (
    fit_df["proportion"]
    .astype(float)
    .sort_values(ascending=False)
    .to_numpy()
)
overall_fit = fit_power_law_rank(overall_vals)
rows.append(
    {
        "distribution": "overall_included_categories",
        "semantic_category": "all",
        **overall_fit,
    }
)

# Semantic-specific distributions
semantic_order_for_fit = [s for s in CDI_SEMANTIC_ORDER if s in set(fit_df["cdi_semantic"].astype(str))]
for sem in semantic_order_for_fit:
    vals = (
        fit_df.loc[fit_df["cdi_semantic"] == sem, "proportion"]
        .astype(float)
        .sort_values(ascending=False)
        .to_numpy()
    )
    sem_fit = fit_power_law_rank(vals)
    rows.append(
        {
            "distribution": f"semantic_{sem}",
            "semantic_category": sem,
            **sem_fit,
        }
    )

powerlaw_summary_df = pd.DataFrame(rows)

out_fit_csv = RESULTS_DIR / f"long_tailed_powerlaw_fits_{FILE_SUFFIX}.csv"
out_fit_txt = RESULTS_DIR / f"long_tailed_powerlaw_fits_{FILE_SUFFIX}.txt"
powerlaw_summary_df.to_csv(out_fit_csv, index=False)
with open(out_fit_txt, "w", encoding="utf-8") as f:
    f.write(powerlaw_summary_df.to_string(index=False))

print(f"Saved power-law fit summary CSV: {out_fit_csv}")
print(f"Saved power-law fit summary TXT: {out_fit_txt}")
display(powerlaw_summary_df)


# Save fit figures (log-log empirical vs fitted)
def _plot_empirical_and_fit(ax, values: np.ndarray, title: str, color: str = "#4C78A8"):
    vals = np.asarray(values, dtype=np.float64)
    vals = vals[np.isfinite(vals)]
    vals = vals[vals > 0]
    vals = np.sort(vals)[::-1]

    n = vals.size
    if n == 0:
        ax.set_title(f"{title} (no data)")
        ax.axis("off")
        return

    ranks = np.arange(1, n + 1, dtype=np.float64)
    fit = fit_power_law_rank(vals)

    ax.plot(ranks, vals, "o", ms=4, alpha=0.8, color=color, label="empirical")

    if np.isfinite(fit["alpha"]) and np.isfinite(fit["coef_c"]):
        fit_curve = fit["coef_c"] * (ranks ** (-fit["alpha"]))
        ax.plot(ranks, fit_curve, "-", lw=2, color="black", label="power-law fit")
        ax.set_title(f"{title}\nalpha={fit['alpha']:.3f}, R2(log)={fit['r2_log']:.3f}, n={fit['n_points']}")
    else:
        ax.set_title(f"{title} (insufficient points, n={fit['n_points']})")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Rank")
    ax.set_ylabel(FRAME_PREVALENCE_YLABEL)
    ax.grid(False)
    ax.legend(frameon=False, fontsize=9)


# Overall figure
fig_overall, ax_overall = plt.subplots(figsize=(7, 5), constrained_layout=True)
_plot_empirical_and_fit(
    ax_overall,
    overall_vals,
    "Overall included categories",
    color="#4C78A8",
)
out_overall_png = FIGURES_DIR / f"long_tailed_powerlaw_overall_{FILE_SUFFIX}.png"
out_overall_pdf = FIGURES_DIR / f"long_tailed_powerlaw_overall_{FILE_SUFFIX}.pdf"
fig_overall.savefig(out_overall_png, dpi=150, bbox_inches="tight")
fig_overall.savefig(out_overall_pdf, bbox_inches="tight")
plt.show()
print(f"Saved figure: {out_overall_png}")
print(f"Saved figure: {out_overall_pdf}")


# Semantic-category grid figure
n_sem = len(semantic_order_for_fit)
if n_sem > 0:
    n_cols = 4 if n_sem >= 8 else 3
    n_rows = int(np.ceil(n_sem / n_cols))
    fig_sem, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(5.5 * n_cols, 4.2 * n_rows),
        constrained_layout=True,
    )
    axes = np.array(axes).reshape(-1)

    for i, sem in enumerate(semantic_order_for_fit):
        sem_vals = (
            fit_df.loc[fit_df["cdi_semantic"] == sem, "proportion"]
            .astype(float)
            .sort_values(ascending=False)
            .to_numpy()
        )
        color = CDI_SEMANTIC_COLORS.get(sem, "#4C78A8")
        _plot_empirical_and_fit(axes[i], sem_vals, sem.replace("_", " "), color=color)

    for j in range(n_sem, len(axes)):
        axes[j].axis("off")

    out_sem_png = FIGURES_DIR / f"long_tailed_powerlaw_semantic_grid_{FILE_SUFFIX}.png"
    out_sem_pdf = FIGURES_DIR / f"long_tailed_powerlaw_semantic_grid_{FILE_SUFFIX}.pdf"
    fig_sem.savefig(out_sem_png, dpi=150, bbox_inches="tight")
    fig_sem.savefig(out_sem_pdf, bbox_inches="tight")
    plt.show()

    print(f"Saved figure: {out_sem_png}")
    print(f"Saved figure: {out_sem_pdf}")

In [ ]:
# 5) Precision vs frame prevalence correlation (valid129 and valid85)
#    Correlates category-level precision from manual validation with
#    category-level frame prevalence (same pools as section 1).

from matplotlib.lines import Line2D
from scipy import stats

PRECISION_CSV = PROJECT_ROOT / "annotation" / "per_class_validation_data.csv"

SET_RUN_ROOTS = {
    "valid85": PREPRINT_DIR / "supplemental_results_valid85cats_04302026",
    "valid129": PREPRINT_DIR / "main_results_valid129s_04302026",
}
SET_RESULTS_DIRS = {k: v / "results" for k, v in SET_RUN_ROOTS.items()}
for out_dir in SET_RESULTS_DIRS.values():
    out_dir.mkdir(parents=True, exist_ok=True)
VALID129_RESULTS_DIR = SET_RESULTS_DIRS["valid129"]

df_precision = pd.read_csv(PRECISION_CSV, usecols=["class", "precision"])
df_precision["category"] = df_precision["class"].astype(str).str.strip().str.lower()
df_precision["precision"] = pd.to_numeric(df_precision["precision"], errors="coerce")
df_precision = df_precision[["category", "precision"]].dropna(subset=["precision"]).copy()

df_cdi_map = pd.read_csv(CDI_CSV, usecols=["uni_lemma", "category"])
df_cdi_map["category"] = df_cdi_map["category"].astype(str).str.strip().str.lower()
df_cdi_map["uni_lemma"] = df_cdi_map["uni_lemma"].astype(str).str.strip().str.lower()
semantic_map = (
    df_cdi_map.drop_duplicates(subset=["uni_lemma"], keep="first")
    .set_index("uni_lemma")["category"]
    .to_dict()
)

df_det_all = load_detections(FRAME_DATA_CSV)


def _included_categories(path: Path) -> list[str]:
    return [
        line.strip().lower()
        for line in path.read_text().splitlines()
        if line.strip()
    ]


def _rescale_sizes(values: pd.Series, min_size: float = 30.0, max_size: float = 220.0) -> np.ndarray:
    arr = values.to_numpy(dtype=float)
    if arr.size == 0:
        return arr
    vmin = float(arr.min())
    vmax = float(arr.max())
    if np.isclose(vmax, vmin):
        return np.full(arr.shape, (min_size + max_size) / 2.0)
    return min_size + ((arr - vmin) / (vmax - vmin)) * (max_size - min_size)


def _corr_for_category_set(set_name: str, included_txt: Path) -> tuple[pd.DataFrame, dict]:
    included = _included_categories(included_txt)
    df_prop, pool_label = build_frame_prevalence_table(
        df_det_all,
        set_name,
        included,
        semantic_map,
    )

    merged = df_prop.merge(df_precision, on="category", how="inner")
    merged = merged.dropna(subset=["precision", "proportion"]).copy()

    pearson_r, pearson_p = stats.pearsonr(merged["precision"], merged["proportion"])
    spearman_rho, spearman_p = stats.spearmanr(merged["precision"], merged["proportion"])

    summary = {
        "category_set": set_name,
        "frame_pool": pool_label,
        "n_categories": int(merged.shape[0]),
        "pearson_r": float(pearson_r),
        "pearson_p": float(pearson_p),
        "spearman_rho": float(spearman_rho),
        "spearman_p": float(spearman_p),
    }
    return merged.sort_values("proportion", ascending=False).reset_index(drop=True), summary


corr_by_set = {}
corr_details = []
corr_summary_rows = []

for set_name, included_txt in CATEGORY_FILES.items():
    merged_set, summary_set = _corr_for_category_set(set_name, included_txt)
    corr_by_set[set_name] = merged_set.copy()

    merged_with_set = merged_set.copy()
    merged_with_set.insert(0, "category_set", set_name)
    corr_details.append(merged_with_set)
    corr_summary_rows.append(summary_set)

corr_details_df = pd.concat(corr_details, ignore_index=True)
corr_summary_df = pd.DataFrame(corr_summary_rows).sort_values("category_set").reset_index(drop=True)

for set_name, merged_set in corr_by_set.items():
    set_results_dir = SET_RESULTS_DIRS[set_name]
    summary_one = corr_summary_df[corr_summary_df["category_set"] == set_name].copy()
    summary_name = f"precision_vs_frame_prevalence_correlation_filtered-0.27_{set_name}.csv"
    details_name = f"precision_vs_frame_prevalence_by_category_filtered-0.27_{set_name}.csv"

    summary_path_set = set_results_dir / summary_name
    details_path_set = set_results_dir / details_name
    summary_one.to_csv(summary_path_set, index=False)
    merged_set.to_csv(details_path_set, index=False)

    summary_path_129 = VALID129_RESULTS_DIR / summary_name
    details_path_129 = VALID129_RESULTS_DIR / details_name
    summary_one.to_csv(summary_path_129, index=False)
    merged_set.to_csv(details_path_129, index=False)

    print(f"Saved summary ({set_name}) -> {summary_path_set}")
    print(f"Saved details ({set_name}) -> {details_path_set}")
    if set_results_dir != VALID129_RESULTS_DIR:
        print(f"Mirrored summary ({set_name}) -> {summary_path_129}")
        print(f"Mirrored details ({set_name}) -> {details_path_129}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
set_order = ["valid85", "valid129"]
for ax, set_name in zip(axes, set_order):
    df_s = corr_by_set[set_name].copy()
    point_sizes = _rescale_sizes(df_s["count_frames"])
    point_colors = [
        CDI_SEMANTIC_COLORS.get(s, CDI_SEMANTIC_COLORS["other"])
        for s in df_s["cdi_semantic"]
    ]

    ax.scatter(
        df_s["precision"],
        df_s["proportion"],
        s=point_sizes,
        alpha=0.82,
        edgecolor="white",
        linewidth=0.4,
        c=point_colors,
    )

    x = df_s["precision"].to_numpy(dtype=float)
    y = df_s["proportion"].to_numpy(dtype=float)
    m, b = np.polyfit(x, y, deg=1)
    xx = np.linspace(x.min(), x.max(), 200)
    ax.plot(xx, m * xx + b, color="#444444", linewidth=2)

    row = corr_summary_df[corr_summary_df["category_set"] == set_name].iloc[0]
    ax.set_title(
        f"{set_name} (n={int(row['n_categories'])})
"
        f"Pearson r={row['pearson_r']:.3f}, p={row['pearson_p']:.2g}
"
        f"Spearman rho={row['spearman_rho']:.3f}, p={row['spearman_p']:.2g}
"
        f"Pool: {row['frame_pool']}; color=CDI semantic; size=frames with category",
        fontsize=11,
    )
    ax.set_xlabel("Category precision")
    ax.set_ylabel(FRAME_PREVALENCE_YLABEL)
    _apply_axis_style(ax)

    sem_present = [s for s in CDI_SEMANTIC_ORDER if s in set(df_s["cdi_semantic"])]
    sem_handles = [
        Line2D(
            [0], [0], marker="o", markersize=7,
            markerfacecolor=CDI_SEMANTIC_COLORS.get(s, CDI_SEMANTIC_COLORS["other"]),
            markeredgecolor="none", linestyle="", label=s.replace("_", " "),
        )
        for s in sem_present
    ]

    counts_unique = np.unique(df_s["count_frames"].to_numpy(dtype=float))
    if counts_unique.size >= 3:
        count_examples = np.quantile(counts_unique, [0.1, 0.5, 0.9]).astype(int)
    else:
        count_examples = counts_unique.astype(int)
    count_examples = np.unique(count_examples)
    size_examples = _rescale_sizes(pd.Series(count_examples.astype(float)))
    size_handles = [
        Line2D(
            [0], [0], marker="o", markersize=np.sqrt(sz),
            markerfacecolor="#777777", markeredgecolor="none", linestyle="",
            label=f"{int(ct)} frames with category",
        )
        for ct, sz in zip(count_examples, size_examples)
    ]

    legend_sem = ax.legend(
        handles=sem_handles, title="CDI semantic", loc="upper left",
        bbox_to_anchor=(1.02, 1.0), fontsize=8, title_fontsize=9, frameon=True,
    )
    ax.add_artist(legend_sem)
    ax.legend(
        handles=size_handles, title="Marker size (frames with category)",
        loc="upper left", bbox_to_anchor=(1.02, 0.55), fontsize=8, title_fontsize=9, frameon=True,
    )

plot_png_name = "precision_vs_frame_prevalence_scatter_valid85_valid129.png"
plot_pdf_name = "precision_vs_frame_prevalence_scatter_valid85_valid129.pdf"

for out_dir in {SET_RESULTS_DIRS["valid85"], VALID129_RESULTS_DIR}:
    png_path = out_dir / plot_png_name
    pdf_path = out_dir / plot_pdf_name
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"Saved scatter figure: {png_path}")
    print(f"Saved scatter figure: {pdf_path}")

plt.show()

corr_summary_df


In [ ]:
# 6) Sensitivity analysis:
#    - Precision-weighted frequency ranking
#    - Ranking after excluding low-precision categories

TOP_K = 10
LOW_PREC_Q = 0.10           # Exclude bottom 10% by precision (set-specific)
LOW_PREC_ABS = 0.70         # Also report exclusion under an absolute threshold

sensitivity_rows = []
ranking_tables = {}

for set_name in ["valid85", "valid129"]:
    df_s = corr_by_set[set_name].copy()

    # Baseline ranking by frame prevalence count.
    df_s = df_s.sort_values("count_frames", ascending=False).reset_index(drop=True)
    df_s["raw_rank"] = np.arange(1, len(df_s) + 1)

    # Precision-weighted frame-count sensitivity view.
    df_s["weighted_count"] = df_s["count_frames"] * df_s["precision"]
    df_s["weighted_prop"] = df_s["weighted_count"] / df_s["weighted_count"].sum()
    df_s = df_s.sort_values("weighted_count", ascending=False).reset_index(drop=True)
    df_s["weighted_rank"] = np.arange(1, len(df_s) + 1)

    df_s = df_s.sort_values("raw_rank").reset_index(drop=True)

    q_threshold = float(df_s["precision"].quantile(LOW_PREC_Q))

    top_raw = (
        df_s.sort_values("count_frames", ascending=False)
        .head(TOP_K)[["category", "count_frames", "precision", "cdi_semantic"]]
        .rename(columns={"count_frames": "raw_count_frames"})
        .reset_index(drop=True)
    )

    top_weighted = (
        df_s.sort_values("weighted_count", ascending=False)
        .head(TOP_K)[["category", "weighted_count", "precision", "cdi_semantic"]]
        .reset_index(drop=True)
    )

    trimmed_q = df_s[df_s["precision"] >= q_threshold].copy()
    top_trimmed_q = (
        trimmed_q.sort_values("count_frames", ascending=False)
        .head(TOP_K)[["category", "count_frames", "precision", "cdi_semantic"]]
        .rename(columns={"count_frames": "trimmed_q_count_frames"})
        .reset_index(drop=True)
    )

    trimmed_abs = df_s[df_s["precision"] >= LOW_PREC_ABS].copy()
    top_trimmed_abs = (
        trimmed_abs.sort_values("count_frames", ascending=False)
        .head(TOP_K)[["category", "count_frames", "precision", "cdi_semantic"]]
        .rename(columns={"count_frames": "trimmed_abs_count_frames"})
        .reset_index(drop=True)
    )

    raw_top_set = set(top_raw["category"])
    weighted_top_set = set(top_weighted["category"])
    trimmed_q_top_set = set(top_trimmed_q["category"])
    trimmed_abs_top_set = set(top_trimmed_abs["category"])

    sensitivity_rows.append(
        {
            "category_set": set_name,
            "top_k": TOP_K,
            "q_threshold_precision": q_threshold,
            "abs_threshold_precision": LOW_PREC_ABS,
            "n_excluded_q": int((df_s["precision"] < q_threshold).sum()),
            "n_excluded_abs": int((df_s["precision"] < LOW_PREC_ABS).sum()),
            "top1_raw": top_raw.iloc[0]["category"] if len(top_raw) else None,
            "top1_weighted": top_weighted.iloc[0]["category"] if len(top_weighted) else None,
            "top1_trimmed_q": top_trimmed_q.iloc[0]["category"] if len(top_trimmed_q) else None,
            "top1_trimmed_abs": top_trimmed_abs.iloc[0]["category"] if len(top_trimmed_abs) else None,
            "overlap_raw_vs_weighted_topk": len(raw_top_set & weighted_top_set),
            "overlap_raw_vs_trimmed_q_topk": len(raw_top_set & trimmed_q_top_set),
            "overlap_raw_vs_trimmed_abs_topk": len(raw_top_set & trimmed_abs_top_set),
        }
    )

    ranking_tables[set_name] = {
        "full": df_s.copy(),
        "top_raw": top_raw.copy(),
        "top_weighted": top_weighted.copy(),
        "top_trimmed_q": top_trimmed_q.copy(),
        "top_trimmed_abs": top_trimmed_abs.copy(),
    }

sensitivity_summary_df = pd.DataFrame(sensitivity_rows)

for set_name, tables in ranking_tables.items():
    set_results_dir = SET_RESULTS_DIRS[set_name]

    out_names = {
        "full": f"precision_sensitivity_full_table_{set_name}.csv",
        "top_raw": f"precision_sensitivity_top{TOP_K}_raw_{set_name}.csv",
        "top_weighted": f"precision_sensitivity_top{TOP_K}_weighted_{set_name}.csv",
        "top_trimmed_q": f"precision_sensitivity_top{TOP_K}_trimmed_q{LOW_PREC_Q:.2f}_{set_name}.csv",
        "top_trimmed_abs": f"precision_sensitivity_top{TOP_K}_trimmed_abs{LOW_PREC_ABS:.2f}_{set_name}.csv",
    }

    for key, filename in out_names.items():
        p_set = set_results_dir / filename
        p_129 = VALID129_RESULTS_DIR / filename
        tables[key].to_csv(p_set, index=False)
        tables[key].to_csv(p_129, index=False)

summary_name = f"precision_sensitivity_summary_top{TOP_K}.csv"
summary_set_path = SET_RESULTS_DIRS["valid85"] / summary_name
summary_129_path = VALID129_RESULTS_DIR / summary_name
sensitivity_summary_df.to_csv(summary_set_path, index=False)
sensitivity_summary_df.to_csv(summary_129_path, index=False)

print(f"Saved sensitivity summary -> {summary_set_path}")
print(f"Saved sensitivity summary -> {summary_129_path}")

display(sensitivity_summary_df)

for _, r in sensitivity_summary_df.iterrows():
    print(
        f"[{r['category_set']}] top-1 raw={r['top1_raw']}, weighted={r['top1_weighted']}, "
        f"trimmed(q)={r['top1_trimmed_q']}, trimmed(abs)={r['top1_trimmed_abs']}; "
        f"Top-{int(r['top_k'])} overlap raw-vs-weighted={int(r['overlap_raw_vs_weighted_topk'])}/"
        f"{int(r['top_k'])}, raw-vs-trimmed(q)={int(r['overlap_raw_vs_trimmed_q_topk'])}/"
        f"{int(r['top_k'])}, raw-vs-trimmed(abs)={int(r['overlap_raw_vs_trimmed_abs_topk'])}/"
        f"{int(r['top_k'])}."
    )
